# Writing your own Neural Network code

In [1]:
import autograd.numpy as np

# custom imports
from runge_preprocessing import x, x_train, x_train_scaled, x_test, y, y_noise, y_train, y_test, layer_dim, activations, activations_derivative, ETA_VALUES, LAMBDA_VALUES, MOMENTUM, RUNGE_MAX_ITERATIONS
from neural_network import NN
import schedulers


In [2]:
# TESTING BLOCK


runge_NN = NN(dims = layer_dim, activation_funcs = activations, activation_ders = activations_derivative)

# Defining optimizers
adam_optimizer = schedulers.ADAM(eta=0.001, rho=0.001, rho2=0.001)
sgd_optimizer = schedulers.momentum(eta=0.001,momentum=0.9)
rmsprop_optimizer = schedulers.RMSprop(eta=0.001,rho=0.1)


# Training and validation
scores, predictions = runge_NN.fit(X=x_train_scaled, t=y_train, X_val=x_test, t_val=y_test, scheduler=adam_optimizer)
print(scores)

#import pandas as pd
#scores = pd.DataFrame(scores)

#print(scores)

"""
suggestions and questions
- rename scores to epoch_scores
- training and validation error depends on chosen cost function, accuracy or mse. Just average values in column?
- added in fit
            if val_set: return scores, pred_val
            else: return scores
- adam optimizer gives nan
"""



Using scheduler: ADAM with Eta=0.001
{'training_errors': array([0.10860084, 0.10242936, 0.09717156, 0.09280614, 0.08895843,
       0.08522044, 0.08160119, 0.07808214, 0.07469046, 0.07139873,
       0.06824972, 0.06523109, 0.06238082, 0.05964838, 0.0570713 ,
       0.05467627, 0.05232213, 0.05029152, 0.04813138, 0.04635026,
       0.04440429, 0.042877  , 0.04110576, 0.03976865, 0.03816837,
       0.03706232, 0.03558944, 0.03465828, 0.03331846, 0.03254315,
       0.03131419, 0.03064981, 0.02953921, 0.02896493, 0.02796328,
       0.02746833, 0.02656266, 0.02614369, 0.02531815, 0.02498013,
       0.02421505, 0.02394898, 0.02323528, 0.02302097, 0.02236144,
       0.02219485, 0.02158278, 0.02144487, 0.02088322, 0.02076332,
       0.02025205, 0.02014692, 0.01968358, 0.01958769, 0.01917159,
       0.01907939, 0.01870917, 0.01862077, 0.01829107, 0.01821347,
       0.01791483, 0.01785818, 0.01757806, 0.01754309, 0.01727452,
       0.01725648, 0.01699883, 0.01699254, 0.01674678, 0.01674775,
     

'\nsuggestions and questions\n- rename scores to epoch_scores\n- training and validation error depends on chosen cost function, accuracy or mse. Just average values in column?\n- added in fit\n            if val_set: return scores, pred_val\n            else: return scores\n- adam optimizer gives nan\n'

In [3]:
import time
from cost_functions import mse
mse_formula = mse().cost

def neural_network_loop(etas, lambdas, optimizer_name, max_iterations, momentum_val=0.9, verbose=True):

    results = []

    for eta in etas:
        for lmbd in lambdas:
            
            if verbose:
                print(f"\nTraining with lr={eta}, lambda={lmbd}, iteration={max_iterations}, optimizer={optimizer_name}")

            start_time = time.time()

            if optimizer_name == 'ADAM':
                optimizer = schedulers.ADAM(eta, rho=lmbd, rho2=lmbd)   # both rho and rho1 at same time? different looping...?
            elif optimizer_name == 'SGD':
                optimizer = schedulers.momentum(eta, momentum=momentum_val)
            elif optimizer_name == 'RMSprop':
                optimizer = schedulers.RMSprop(eta=etas,rho=lmbd)

            epoch_scores, predictions = runge_NN.fit(X=x_train_scaled, t=y_train, X_val=x_test, t_val=y_test, epochs=max_iterations, scheduler=optimizer)
            mse =  mse_formula(y_true=y_test, y_pred=predictions)
            end_time = time.time()
            elapsed_time = end_time - start_time

            results.append({
                'Learning Rate': eta,
                'Lambda': lmbd,
                'Iterations': max_iterations,
                'Elapsed time': elapsed_time,
                'epoch_training_errors': epoch_scores['training_errors'],
                'epoch_validation_errors': epoch_scores['validation_errors'],
                'mse': mse,
                'predictions': predictions
            })
    return pd.DataFrame(results)

results_sgd = neural_network_loop(ETA_VALUES, LAMBDA_VALUES, 'SGD', RUNGE_MAX_ITERATIONS, momentum_val=MOMENTUM, verbose=False)
results_rmsprop = neural_network_loop(ETA_VALUES, LAMBDA_VALUES, 'RMSprop', RUNGE_MAX_ITERATIONS, momentum_val=MOMENTUM, verbose=False)
#results_adam = neural_network_loop(ETA_VALUES, LAMBDA_VALUES, 'ADAM', RUNGE_MAX_ITERATIONS, momentum_val=MOMENTUM, verbose=True) - optimizer does not work - NOT MOMENTUM




Using scheduler: momentum with Eta=0.1
Using scheduler: momentum with Eta=0.1
Using scheduler: momentum with Eta=0.1
Using scheduler: momentum with Eta=0.1
Using scheduler: momentum with Eta=0.1
Using scheduler: momentum with Eta=0.1
Using scheduler: momentum with Eta=0.1
Using scheduler: momentum with Eta=0.1
Using scheduler: momentum with Eta=0.1
Using scheduler: momentum with Eta=0.1
Using scheduler: momentum with Eta=0.01
Using scheduler: momentum with Eta=0.01
Using scheduler: momentum with Eta=0.01
Using scheduler: momentum with Eta=0.01
Using scheduler: momentum with Eta=0.01
Using scheduler: momentum with Eta=0.01
Using scheduler: momentum with Eta=0.01
Using scheduler: momentum with Eta=0.01
Using scheduler: momentum with Eta=0.01
Using scheduler: momentum with Eta=0.01
Using scheduler: momentum with Eta=0.001
Using scheduler: momentum with Eta=0.001
Using scheduler: momentum with Eta=0.001
Using scheduler: momentum with Eta=0.001
Using scheduler: momentum with Eta=0.001
Using

NameError: name 'pd' is not defined

In [ ]:
from plotting import plot_heatmap

SHOW_PLOT = False

plot_heatmap(results_sgd, title='Runge function with SGD', heat_metric='mse', filename=f'runge_heatmap_sgd_mse_iter{RUNGE_MAX_ITERATIONS}_momentum{MOMENTUM}', show_plot=SHOW_PLOT)
plot_heatmap(results_rmsprop, title='Runge function with RMSprop', heat_metric='mse', filename=f'runge_heatmap_rmsprop_mse_iter{RUNGE_MAX_ITERATIONS}_momentum{MOMENTUM}', show_plot=SHOW_PLOT)
#plot_heatmap(results_adam, title='Runge function with ADAM', heat_metric='mse') - adam does not work



In [ ]:
from plotting import plot_runges

SHOW_PLOT = True

# choose which dataset to plot
min_row = results_sgd.loc[results_sgd['mse'].idxmin()]
predictions = min_row['predictions'] # find predictions for lowest mse
plot_runges(x, y, x_test, predictions, title=f'Runge function - SGD', filename=f'runge_predicted_iter{RUNGE_MAX_ITERATIONS}_momentum{MOMENTUM}', show_plot=SHOW_PLOT)


# choose which dataset to plot
min_row = results_rmsprop.loc[results_sgd['mse'].idxmin()]
predictions = min_row['predictions'] # find predictions for lowest mse
plot_runges(x, y, x_test, predictions, title=f'Runge function - RMSprop', filename=f'runge_predicted_rmsprop_iter{RUNGE_MAX_ITERATIONS}_momentum{MOMENTUM}', show_plot=SHOW_PLOT)

# ADAM - choose which dataset to plot     DOES NOT WORK
#min_row = results_adam.loc[results_sgd['mse'].idxmin()]
#predictions = min_row['predictions'] # find predictions for lowest mse
#plot_runges(x, y, x_test, predictions, title=f'Runge function - ADAM', filename=f'runge_predicted_adam_iter{RUNGE_MAX_ITERATIONS}_momentum{MOMENTUM}', show_plot=SHOW_PLOT) - optimizer does not work - NOT MOMENTUM